<a href="https://colab.research.google.com/github/stevenolanecon/7002LBSAI/blob/main/notebooks/week8_online_walkthrough.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Week 8 online session – count outcomes

Companion notebook to the Week 8 online recording. Run each cell as you watch: the numbers on the slides are computed here.

Data: 600 customer accounts of a software company over the last six months, generated for this module. How many support tickets does each account raise?

**Before you start:**

- Go to **File → Save a copy in Drive**. Until you do this, your work only exists in this browser tab – you can't save changes back to GitHub, and closing the tab or losing the session will lose your work.
- Turn off Colab's autocomplete: **Tools → Settings → Editor → uncheck "Show context-powered code completions."** It's a setting on your own Google account, not something built into this notebook, so you'll need to do it once per account.

## Setup

In [ ]:
import warnings
warnings.filterwarnings("ignore")   # negative binomial prints harmless warnings while it searches

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy import stats

data_path = "https://raw.githubusercontent.com/stevenolanecon/7002LBSAI/main/data/support_tickets.csv"
df = pd.read_csv(data_path)
df.head()

## A first look at the counts

In [ ]:
print(df["tickets"].describe())
print("Share with no tickets:", round((df["tickets"] == 0).mean(), 3))
print("Accounts active for less than six months:", (df["months_active"] < 6).sum())

df["tickets"].value_counts().sort_index().plot(kind="bar", xlabel="Tickets", ylabel="Accounts")
plt.show()

## Why not OLS?

OLS can predict a negative number of tickets.

In [ ]:
ols_model = smf.ols("tickets ~ users + trained + plan", data=df).fit()
print("Negative predictions:", (ols_model.fittedvalues < 0).sum())
print("Lowest prediction:", round(ols_model.fittedvalues.min(), 2))

## Poisson regression, with the exposure

`exposure=` adds ln(months active) to the model with its coefficient fixed at 1 (an offset), so we model tickets per month. `plan` is text, so Basic is the reference category.

In [ ]:
poisson_model = smf.glm("tickets ~ users + trained + plan", data=df,
                        family=sm.families.Poisson(),
                        exposure=df["months_active"]).fit()
print(poisson_model.summary())

## What if we left the exposure out?

In [ ]:
no_exposure = smf.glm("tickets ~ users + trained + plan", data=df,
                      family=sm.families.Poisson()).fit()
print("trained IRR with exposure:   ", round(np.exp(poisson_model.params["trained"]), 3))
print("trained IRR without exposure:", round(np.exp(no_exposure.params["trained"]), 3))

# Why: newer accounts were more likely to be trained
df.groupby(df["months_active"] < 6)["trained"].mean()

## Incidence rate ratios

In [ ]:
# statsmodels has no IRR option, so exponentiate the coefficients and their CI together
irr = np.exp(pd.concat([poisson_model.params, poisson_model.conf_int()], axis=1))
irr.columns = ["IRR", "2.5%", "97.5%"]
irr.round(3)

### Reading them

- **Seats:** each extra seat multiplies expected tickets by the `users` IRR. For 10 seats, raise it to the power 10.
- **Training:** an IRR below 1 means fewer tickets; 1 − IRR is the percentage drop.
- **Plan:** each IRR compares that plan with Basic. Divide two IRRs to compare the plans with each other.

In [ ]:
print("10 more seats:", round(np.exp(10 * poisson_model.params["users"]), 3))
print("Training: % change", round((np.exp(poisson_model.params["trained"]) - 1) * 100, 1))
print("Enterprise vs Pro:", round(np.exp(poisson_model.params["plan[T.Enterprise]"] - poisson_model.params["plan[T.Pro]"]), 2))

## A predicted count for one account

In [ ]:
account = pd.DataFrame({"users": [50], "trained": [0], "plan": ["Pro"]})
print("Six months, untrained:", round(poisson_model.predict(account, exposure=[6]).iloc[0], 2))
print("One month, untrained: ", round(poisson_model.predict(account, exposure=[1]).iloc[0], 2))
print("Six months, trained:  ", round(poisson_model.predict(account.assign(trained=1), exposure=[6]).iloc[0], 2))

## Is Poisson a good fit? The dispersion statistic

If Poisson is right, an account predicted to raise μ̂ tickets misses by about √μ̂. So each squared miss divided by μ̂ should average about 1. Well above 1 means overdispersion.

In [ ]:
mu = poisson_model.fittedvalues            # predicted tickets per account
((df["tickets"] - mu) ** 2 / mu).mean()    # about 1 if Poisson is right

statsmodels has a shortcut for (almost) the same number. It divides by the residual degrees of freedom instead of the number of accounts.

In [ ]:
poisson_model.pearson_chi2 / poisson_model.df_resid

## You can see it in the zeros

The share of accounts at each count, against what each model predicts.

In [ ]:
nb_model = smf.negativebinomial("tickets ~ users + trained + plan", data=df,
                                exposure=df["months_active"]).fit(disp=0)

mu_p = poisson_model.predict(df, exposure=df["months_active"])
mu_nb = nb_model.predict(df, exposure=df["months_active"])
alpha = nb_model.params["alpha"]

k = np.arange(0, 11)
observed = [(df["tickets"] == j).mean() for j in k]
poisson_share = [stats.poisson.pmf(j, mu_p).mean() for j in k]
nb_share = [stats.nbinom.pmf(j, 1 / alpha, 1 / (1 + alpha * mu_nb)).mean() for j in k]

plt.bar(k, observed, alpha=0.5, label="Observed")
plt.plot(k, poisson_share, "o-", label="Poisson")
plt.plot(k, nb_share, "o-", label="Negative binomial")
plt.xlabel("Tickets"); plt.ylabel("Share of accounts"); plt.legend()
plt.show()

## Negative binomial

In [ ]:
print(nb_model.summary())

## Standard errors and IRRs, side by side

In [ ]:
terms = ["users", "trained", "plan[T.Pro]", "plan[T.Enterprise]"]
pd.DataFrame({"Poisson SE": poisson_model.bse[terms],
              "Neg. binomial SE": nb_model.bse[terms],
              "Ratio": nb_model.bse[terms] / poisson_model.bse[terms],
              "Poisson IRR": np.exp(poisson_model.params[terms]),
              "Neg. binomial IRR": np.exp(nb_model.params[terms])}).round(3)